# 04 — Land Use Mix

Computes diversity metrics of PLUTO land use distribution per census tract. Uses Shannon entropy and HHI (Herfindahl-Hirschman Index) to measure how mixed vs. homogeneous each zone is.

**Data source:** PLUTO CSV (NYC only — `needs_pluto`).

**Note:** Raw area ratios (`comarea`, `resarea`, etc.) are intentionally excluded to prevent Y variable leakage. Only mix/diversity metrics are kept.

**Output columns:** `tract_id`, `landuse_entropy`, `landuse_hhi`

**Output file:** `csv/04_land_use_mix.csv`

In [ ]:
ZONES_CONFIG = "zones.json"

In [ ]:
import pandas as pd
import numpy as np
import json
import os

os.makedirs("csv", exist_ok=True)

with open(ZONES_CONFIG, encoding="utf-8") as f:
    config = json.load(f)

if not config["feature_flags"].get("needs_pluto", False):
    print("PLUTO not available — skipping notebook 04.")
    df_tracts = pd.read_csv("csv/01_zone_definition.csv", dtype={"tract_id": str})
    df_empty = pd.DataFrame({"tract_id": df_tracts["tract_id"]})
    for col in ["landuse_entropy", "landuse_hhi"]:
        df_empty[col] = np.nan
    df_empty.to_csv("csv/04_land_use_mix.csv", index=False)
    raise SystemExit("Skipped — needs_pluto=false")

PLUTO_PATH = config["pluto_path"]
BOROUGH_CODES = config["borough_codes"]
BOROUGH_FILTER = config["borough_filter"]
boro_code_filter = [str(BOROUGH_CODES[b]) for b in BOROUGH_FILTER]
print(f"Loading PLUTO from {PLUTO_PATH}")

In [ ]:
# ── Load PLUTO ────────────────────────────────────────
COLS = ["borocode", "bct2020", "landuse", "lotarea"]

df_pluto = pd.read_csv(PLUTO_PATH, usecols=COLS, dtype={"bct2020": str, "landuse": str})
df_pluto = df_pluto[df_pluto["borocode"].astype(str).isin(boro_code_filter)].copy()
df_pluto["lotarea"] = pd.to_numeric(df_pluto["lotarea"], errors="coerce").fillna(0)

print(f"PLUTO rows: {len(df_pluto):,}")
print(f"Unique landuse codes: {df_pluto['landuse'].nunique()}")

In [ ]:
# ── Compute entropy and HHI per tract ─────────────────

def shannon_entropy(proportions):
    """Shannon entropy from a list of proportions (0–1)."""
    proportions = proportions[proportions > 0]
    if len(proportions) == 0:
        return 0.0
    return -np.sum(proportions * np.log2(proportions))


def herfindahl_hirschman(proportions):
    """HHI from proportions. 1.0 = perfectly concentrated, 1/N = perfectly diverse."""
    return np.sum(proportions ** 2)


records = []

for tract_id, group in df_pluto.groupby("bct2020"):
    total_area = group["lotarea"].sum()
    if total_area == 0:
        records.append({"tract_id": tract_id, "landuse_entropy": 0.0, "landuse_hhi": 1.0})
        continue
    
    # Area-weighted landuse distribution
    lu_area = group.groupby("landuse")["lotarea"].sum()
    proportions = (lu_area / total_area).values
    
    records.append({
        "tract_id": tract_id,
        "landuse_entropy": round(shannon_entropy(proportions), 4),
        "landuse_hhi": round(herfindahl_hirschman(proportions), 4),
    })

df_mix = pd.DataFrame(records)
print(f"Computed mix metrics for {len(df_mix)} tracts")
print(f"\nEntropy: mean={df_mix['landuse_entropy'].mean():.3f}, "
      f"min={df_mix['landuse_entropy'].min():.3f}, "
      f"max={df_mix['landuse_entropy'].max():.3f}")
print(f"HHI:     mean={df_mix['landuse_hhi'].mean():.3f}, "
      f"min={df_mix['landuse_hhi'].min():.3f}, "
      f"max={df_mix['landuse_hhi'].max():.3f}")

In [ ]:
# ── Save output ───────────────────────────────────────
output_path = "csv/04_land_use_mix.csv"
df_mix.to_csv(output_path, index=False, encoding="utf-8")
print(f"Saved: {output_path}  ({len(df_mix)} rows x {df_mix.shape[1]} cols)")
df_mix.head(10)